In [100]:
# Import the pandas library
import pandas as pd

# Read the CSV file into a DataFrame
df = pd.read_csv('./data/Project2_dataset.csv')

In [101]:
# See the shape of the DataFrame
df.shape

(57301, 9)

In [102]:
# Check columns
df.columns

Index(['Airline Name', 'Airline Country', 'Departure Airport Name',
       'Departure Airport City', 'Departure Airport Country/Region',
       'Arrival Airport Name', 'Arrival Airport City',
       'Arrival Airport Country/Region', 'Plane Name'],
      dtype='str')

This data seems to be containing information regarding a flight's information, containing information regarding its airline, the departure and arrival airports, and its name.

We can break this data down into a few datasets:
- Airline information; name and country
- Airport information; name, city, country
- Plane information; its name and the airline its operating under
- The routes that the planes fly on


In [103]:
# Display the first few rows of the DataFrame
df.head()

,Airline Name,Airline Country,Departure Airport Name,Departure Airport City,Departure Airport Country/Region,Arrival Airport Name,Arrival Airport City,Arrival Airport Country/Region,Plane Name
0,Tuna Aero,Sweden,Norsup Airport,Norsup,Vanuatu,Bauerfield International Airport,Port-vila,Vanuatu,Harbin Yunshuji Y12;De Havilland Canada DHC-6 ...
1,Tuna Aero,Sweden,Santo Pekoa International Airport,Santo,Vanuatu,Lonorore Airport,Lonorore,Vanuatu,Harbin Yunshuji Y12;De Havilland Canada DHC-6 ...
2,Tuna Aero,Sweden,Craig Cove Airport,Craig Cove,Vanuatu,Norsup Airport,Norsup,Vanuatu,Harbin Yunshuji Y12;De Havilland Canada DHC-6 ...
3,Tuna Aero,Sweden,Bauerfield International Airport,Port-vila,Vanuatu,Norsup Airport,Norsup,Vanuatu,Harbin Yunshuji Y12;De Havilland Canada DHC-6 ...
4,Tuna Aero,Sweden,Aneityum Airport,Anelghowhat,Vanuatu,Tanna Airport,Tanna,Vanuatu,Harbin Yunshuji Y12;Pilatus Britten-Norman BN-...


## Check missing values and other anomalies

### Null values

In [104]:
# Check null values in the DataFrame
df.isnull().sum()

Airline Name                        0
Airline Country                     0
Departure Airport Name              0
Departure Airport City              0
Departure Airport Country/Region    0
Arrival Airport Name                0
Arrival Airport City                0
Arrival Airport Country/Region      0
Plane Name                          0
dtype: int64

The data seems to not have any missing values. Which is good.

### Duplicated values

In [105]:
# Check duplicated rows in the DataFrame
df.duplicated().sum()

np.int64(0)

The data does not have any duplicated rows.

In [106]:
# Check the number of unique values in each column
df.nunique()

Airline Name                         488
Airline Country                      106
Departure Airport Name              2784
Departure Airport City              2697
Departure Airport Country/Region     223
Arrival Airport Name                2789
Arrival Airport City                2700
Arrival Airport Country/Region       220
Plane Name                          2728
dtype: int64

## Cleaning

### Check for whitespace issues
First we will see if there are any columns with white spaces in the values.

In [107]:
df.apply(lambda col:col.str.strip().ne(col).sum())

Airline Name                        0
Airline Country                     0
Departure Airport Name              0
Departure Airport City              4
Departure Airport Country/Region    0
Arrival Airport Name                0
Arrival Airport City                5
Arrival Airport Country/Region      0
Plane Name                          0
dtype: int64

- `col.str.strip()` - removes leading/trailing spaces from every value in a column
- `.ne(col)` - computes the stripped version to the original
- `.sum()` - counts how many values were different to the original
- `df.apply(lambda col:...)` - runs the check on every column.

Fix:

In [108]:
df['Departure Airport City'] = df['Departure Airport City'].str.strip()
df['Arrival Airport City'] = df['Arrival Airport City'].str.strip()

### Check for case inconsistencies


In [109]:
df['Departure Airport City'].value_counts().head(20)

Departure Airport City
London               1085
Atlanta               759
Paris                 676
Shanghai              601
Beijing               599
Moscow                586
New York              559
Chicago               520
Istanbul              489
Frankfurt             450
Dallas-Fort Worth     449
Los Angeles           423
Tokyo                 410
Singapore             392
Bangkok               382
Barcelona             378
Seoul                 373
Miami                 364
Rome                  358
Amsterdam             358
Name: count, dtype: int64

In [110]:
# Check if stripped version differs from original (catches hidden spaces within names)
name_cols = ['Airline Name', 'Departure Airport Name', 'Arrival Airport Name', 'Plane Name']

for col in name_cols:
    stripped = df[col].str.strip()
    diff = df[col].ne(stripped).sum()
    print(f"{col}: {diff} values with whitespace issues")

Airline Name: 0 values with whitespace issues
Departure Airport Name: 0 values with whitespace issues
Arrival Airport Name: 0 values with whitespace issues
Plane Name: 0 values with whitespace issues


1. Puts the name of the columns in an array.
2. Loops through each index of the array
3. In each loop, access the column in `df` that has the same name as `col` and apply some things

Based on this snippet, no `name` related columns have case issues.

### Export Cleaned data

In [111]:
df.to_csv('./csv_output/Project2_dataset_cleaned.csv', index=False)

## Exploring Each Column

Let's import cleaned data first

In [112]:
df_cleaned = pd.read_csv('./csv_output/Project2_dataset_cleaned.csv')

### Countries with the most airlines

In [113]:
df_cleaned['Airline Country'].value_counts()

Airline Country
United States          11207
United Kingdom          5704
Mexico                  3721
Greece                  2707
Russia                  2688
                       ...  
Uruguay                    8
Trinidad and Tobago        8
Afghanistan                6
Ireland                    5
Belarus                    4
Name: count, Length: 106, dtype: int64

### 10 Countries with the most flight coming out 

In [114]:
df_cleaned['Departure Airport Country/Region'].value_counts().head(10)

Departure Airport Country/Region
United States     9366
China             7933
United Kingdom    2236
Spain             2127
Germany           2048
France            1807
Russia            1712
Italy             1679
India             1342
Brazil            1317
Name: count, dtype: int64

### 10 Countries with the most flight coming in

In [115]:
df_cleaned['Arrival Airport Country/Region'].value_counts().head(10)

Arrival Airport Country/Region
United States    9262
China            7933
Canada           2414
Spain            2229
Germany          2041
France           1804
Russia           1738
Italy            1682
Brazil           1308
India            1287
Name: count, dtype: int64

### Domestic FLights

In [116]:
df_cleaned[df['Departure Airport Country/Region'] == df['Arrival Airport Country/Region']].value_counts('Departure Airport Country/Region').head(10)

Departure Airport Country/Region
United States    7046
China            6778
Brazil           1101
India             918
Russia            892
Canada            646
Indonesia         607
Australia         558
Spain             548
Japan             500
Name: count, dtype: int64

### International Flight

In [ ]:
df_cleaned[df_cleaned['Departure Airport Country/Region'] != df_cleaned['Arrival Airport Country/Region']].value_counts('Departure Airport Country/Region').head(10)

Departure Airport Country/Region
United States           2320
United Kingdom          1987
Germany                 1863
Spain                   1579
France                  1325
Italy                   1275
China                   1155
Russia                   820
United Arab Emirates     643
Japan                    632
Name: count, dtype: int64

In [ ]:
df_cleaned.columns

Index(['Airline Name', 'Airline Country', 'Departure Airport Name',
       'Departure Airport City', 'Departure Airport Country/Region',
       'Arrival Airport Name', 'Arrival Airport City',
       'Arrival Airport Country/Region', 'Plane Name'],
      dtype='str')

## Creating Nodes
CSVs that will be made from this dataset are:
- Airline
- Airport
- 

### Airline Node
The airline information needed from the dataset are `Airline Name` and `Airline Country`. 

In [ ]:
# Create a new DataFrame with unique combinations of 'Airline Country' and 'Airline Name'
airlines = df_cleaned[['Airline Name', 'Airline Country']].drop_duplicates().reset_index(drop=True)

- `[['Airline Name','Airline Country']]` - take only these two airline columns
- `drop_duplicates()` - remove duplicates of the `Airline Name-Airline Country` 
- `reset_index(drop=True)` - clean up index after removing duplicates


In [120]:
# Rename the columns of the new DataFrame
airlines.columns = ['name', 'country']

This renames the new columns for the Airline dataset.

In [121]:
# Save the DataFrame to a CSV file
airlines.to_csv('./csv_output/airlines.csv', index=False)

In [122]:
airlines

,name,country
0,Tuna Aero,Sweden
1,Cranwell Flying Training Unit,United Kingdom
2,Renan,Moldova
3,Voar Lda,Angola
4,Southern Cargo Air Lines,Russia
...,...,...
483,Wings Airways,United States
484,Samson Aviation,United Kingdom
485,Rynes Aviation,United States
486,VIP Air,Slovakia


### Aiport Node

In [146]:
#
dep_airport = df_cleaned[['Departure Airport Name','Departure Airport City', 'Departure Airport Country/Region']].rename(columns={'Departure Airport Name': 'name', 'Departure Airport City': 'city', 'Departure Airport Country/Region': 'country'} )
arr_airport = df_cleaned[['Arrival Airport Name','Arrival Airport City', 'Arrival Airport Country/Region']].rename(columns={'Arrival Airport Name': 'name', 'Arrival Airport City': 'city', 'Arrival Airport Country/Region': 'country'})  

airports = pd.concat([dep_airport, arr_airport]).drop_duplicates().reset_index(drop=True)
airports.to_csv('./csv_output/airports.csv', index=False)

### Routes
This one is actually a relationship, not a node, between departure airport and arrival airport

In [ ]:
# Create a new DataFrame with unique combinations of 'Departure Airport Name' and 'Arrival Airport Name'
routes = df_cleaned[['Departure Airport Name', 'Arrival Airport Name',]].rename(columns={'Departure Airport Name': 'departure_airport', 'Arrival Airport Name': 'arrival_airport'}).reset_index(drop=True)
routes.to_csv('./csv_output/routes.csv', index=False)

### Operates Node
This is also a relationship between an airline, the airports they operate in, and the planes they have.

In [135]:
# Create a new DataFrame with unique combinations of 'Airline Name', 'Departure Airport Name', 'Arrival Airport Name', and 'Plane Name'
operates = df_cleaned[['Airline Name', 'Departure Airport Name', 'Arrival Airport Name','Plane Name']].rename(columns={'Airline Name': 'airline_name', 'Departure Airport Name': 'departure_airport', 'Arrival Airport Name': 'arrival_airport', 'Plane Name': 'plane_name'}).drop_duplicates().reset_index(drop=True)
operates.to_csv('./csv_output/operates.csv', index=False)

In [129]:
df_cleaned.columns

Index(['Airline Name', 'Airline Country', 'Departure Airport Name',
       'Departure Airport City', 'Departure Airport Country/Region',
       'Arrival Airport Name', 'Arrival Airport City',
       'Arrival Airport Country/Region', 'Plane Name'],
      dtype='str')

In [139]:
airlines.head()

,name,country
0,Tuna Aero,Sweden
1,Cranwell Flying Training Unit,United Kingdom
2,Renan,Moldova
3,Voar Lda,Angola
4,Southern Cargo Air Lines,Russia


In [147]:
airports.head()

,name,city,country
0,Norsup Airport,Norsup,Vanuatu
1,Santo Pekoa International Airport,Santo,Vanuatu
2,Craig Cove Airport,Craig Cove,Vanuatu
3,Bauerfield International Airport,Port-vila,Vanuatu
4,Aneityum Airport,Anelghowhat,Vanuatu


In [143]:
routes.head()

,departure_airport,arrival_airport
0,Norsup Airport,Bauerfield International Airport
1,Santo Pekoa International Airport,Lonorore Airport
2,Craig Cove Airport,Norsup Airport
3,Bauerfield International Airport,Norsup Airport
4,Aneityum Airport,Tanna Airport


In [144]:
operates.head()

,airline_name,departure_airport,arrival_airport,plane_name
0,Tuna Aero,Norsup Airport,Bauerfield International Airport,Harbin Yunshuji Y12;De Havilland Canada DHC-6 ...
1,Tuna Aero,Santo Pekoa International Airport,Lonorore Airport,Harbin Yunshuji Y12;De Havilland Canada DHC-6 ...
2,Tuna Aero,Craig Cove Airport,Norsup Airport,Harbin Yunshuji Y12;De Havilland Canada DHC-6 ...
3,Tuna Aero,Bauerfield International Airport,Norsup Airport,Harbin Yunshuji Y12;De Havilland Canada DHC-6 ...
4,Tuna Aero,Aneityum Airport,Tanna Airport,Harbin Yunshuji Y12;Pilatus Britten-Norman BN-...


In [148]:
print(operates['plane_name'].iloc[0])

Harbin Yunshuji Y12;De Havilland Canada DHC-6 Twin Otter;Pilatus Britten-Norman BN-2A/B Islander
